In [ ]:
%pip install -q torch_geometric

### Complete Flow
- Encode
- Decode
- Calculate loss
- Update parameters
- Evaluate

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score,average_precision_score

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)

c:\Users\USER\Desktop\python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\Desktop\python\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device: cpu


In [ ]:
# datatset=Planetoid(root="./data",name="Cora",transform=T.NormalizeFeatures())
# data=dataset[0]
# torch.save(data,"cora_graph.pt")
# print("Saved Dataset")

In [2]:
data = torch.load(
    "cora_graph.pt",
    map_location="cpu",
    weights_only=False
)

print(data)
print("Node:", data.num_nodes)
print("Edge: ",data.edge_index.shape[1])
print("Features:", data.x.shape)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Node: 2708
Edge:  10556
Features: torch.Size([2708, 1433])


In [3]:
split_transform=T.RandomLinkSplit(num_val=0.05,num_test=0.1,is_undirected=True,add_negative_train_samples=False)

train_data,val_data,test_data=split_transform(data)
print(train_data)

Data(x=[2708, 1433], edge_index=[2, 8976], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[4488], edge_label_index=[2, 4488])


In [4]:
print(train_data.edge_label_index[0])
print(train_data.edge_label_index[1])

tensor([ 133,  687,  420,  ...,  109,  995, 1899])
tensor([1777, 1358, 1869,  ..., 1772, 1309, 2510])


edge_index=[2, 8976] = no of edges ,which one are connected

In [5]:
print(val_data)

Data(x=[2708, 1433], edge_index=[2, 8976], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[526], edge_label_index=[2, 526])


In [6]:
print(test_data)

Data(x=[2708, 1433], edge_index=[2, 9502], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[1054], edge_label_index=[2, 1054])


In [ ]:
train_data=train_data.to(device)
val_data=val_data.to(device)
test_data=test_data.to(device)
all_original_edges=data.edge_index.to(device)

In [10]:
class GCNLinkPredictor(nn.Module):
    def __init__(self):
            super().__init__()
            
            self.conv1=GCNConv(1433,128)
            self.conv2=GCNConv(128,64)
            
    def encode(self,x,edge_index):
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=F.dropout(x,p=0.3)
        
        z=self.conv2(x,edge_index)
        return z
    
    def decode(self,z,edge_label_index):
        source=edge_label_index[0]
        destination=edge_label_index[1]
        scores=(z[source]*z[destination]).sum(dim=-1)
        return scores

In [11]:
model=GCNLinkPredictor().to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
criterion=nn.BCEWithLogitsLoss()
print(model)

GCNLinkPredictor(
  (conv1): GCNConv(1433, 128)
  (conv2): GCNConv(128, 64)
)


In [15]:
def train():
    model.train()
    optimizer.zero_grad()
    
    #generate node embedding for train datatset
    z=model.encode(train_data.x,train_data.edge_index)
    
    # Positive edges
    pos_edge_index=train_data.edge_label_index
    pos_scores=model.decode(z,pos_edge_index)
    
    # Negative edges
    # #virtually connecting each node as given
    neg_edge_index=negative_sampling(
        edge_index=all_original_edges,
        num_nodes=data.num_nodes,
        num_neg_samples=pos_edge_index.size(1),
        method="sparse"
    )
    neg_scores=model.decode(z,neg_edge_index)
    
    # Combine predictions and labels
    scores=torch.cat([pos_scores,neg_scores])
    
    labels=torch.cat([torch.ones_like(pos_scores),torch.zeros_like(neg_scores)])
    loss=criterion(scores,labels)
    loss.backward()
    optimizer.step()
    return loss.item()

In [16]:
# Disable gradient calculation because we are only evaluating
# This saves memory and computation
@torch.no_grad()
def evaluate(split_data):
    # Set the model to evaluation mode
    # Dropout is disabled during evaluation
    model.eval()
    
    # Generate node embeddings using the graph for this split
    z=model.encode(split_data.x,split_data.edge_index)
    
    # Predict scores for the candidate edges in this split
    scores=model.decode(z,split_data.edge_label_index)
    
    # Convert raw scores (logits) into probabilities between 0 and 1
    probabilities=torch.sigmoid(scores)
    
    # Get the true labels for the candidate edges
    # 1 = positive edge, 0 = negative edge
    labels=split_data.edge_label
    
    # Convert PyTorch tensors into NumPy arrays for sklearn metrics
    y_true=labels.cpu().numpy()
    y_score=probabilities.cpu().numpy()
    
    # ROC-AUC measures how well the model ranks positive edges
    # above negative edges; 1.0 is perfect and 0.5 is random ranking
    auc=roc_auc_score(y_true,y_score)
    
    # Average Precision summarizes precision-recall performance
    # Higher values generally indicate better positive-edge retrieval
    ap=average_precision_score(y_true,y_score)
    
    # Return the two evaluation metrics
    return auc,ap

In [18]:
epochs=10
for epoch in range(1,epochs+1):
    # Train the model and obtain its training loss
    loss=train()
    
    # Evaluate model performance on the validation dataset
    val_auc,val_ap=evaluate(val_data)
    
    if (epoch%10==0):
        print(f"Epoch {epoch:03d} Loss: {loss:.4f} Val AUC: {val_auc}")

# Evaluate the final model on the test dataset
test_auc,test_ap=evaluate(test_data)

# Display the final test results
print("\nFinal Test Results")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(f"Test Average Precision: {test_ap:.4f}")

Epoch 010 Loss: 0.6389 Val AUC: 0.7313970131128107

Final Test Results
Test ROC-AUC: 0.7080
Test Average Precision: 0.7181
